In [76]:
from utils.paths import SAMPLING_DIR, RAW_DIR,PREDICTIONS_DIR

In [77]:
import numpy as np
import pandas as pd
import pickle
from models_scripts.AdaptiveTransferKernel import AdaptiveTransferKernel

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [78]:
comp_space = pd.read_csv(RAW_DIR/"CuNiAl_descriptors.csv")

Experimental data GP - Random splitting

In [79]:
with open(SAMPLING_DIR/"random_split", "rb") as f:
    random_split = pickle.load(f)

In [67]:
X_train_random = random_split['X_train_split'].drop(columns=['r','r_ave','S'])
X_test_random = random_split['X_test_split'].drop(columns=['r','r_ave','S'])

In [84]:
random_split['X_train_split']

,Cu,Ni,Al,r,r_ave,del_r,del_EN,S,VEC
10251,0.500,0.500,0.000,126.000,126.000,0.015873,0.005000,5.762826,1.500
14415,0.475,0.475,0.050,126.850,126.850,0.033004,0.064478,7.125140,1.575
1923,0.950,0.000,0.050,128.750,128.750,0.025392,0.063204,1.650456,1.100
23784,0.025,0.025,0.950,142.150,142.150,0.026254,0.064303,1.938597,2.925
6087,0.000,0.050,0.950,142.050,142.050,0.029151,0.065383,1.650456,2.950
2964,0.000,0.950,0.050,124.950,124.950,0.033141,0.065383,1.650456,2.050
13374,0.640,0.190,0.165,129.075,129.075,0.047524,0.108899,7.469807,1.515
16497,0.500,0.000,0.500,135.500,135.500,0.055351,0.145000,5.762826,2.000
4005,0.050,0.950,0.000,124.200,124.200,0.007019,0.002179,1.650456,1.950
5046,0.050,0.000,0.950,142.250,142.250,0.022982,0.063204,1.650456,2.900


In [68]:
X_train = np.asarray(X_train_random, dtype=float)
y_train = np.asarray(random_split['y_train'], dtype=float).ravel()

kernel = (
    C(1.0, (1e-3, 1e3)) *
    Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e2), nu=2.5)
    + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-8, 1e1))
)

gpr_nt_ran = GaussianProcessRegressor(
    kernel=kernel,
    alpha=0.0,   
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

gpr_nt_ran.fit(X_train, y_train)
print("Learned kernel:", gpr_nt_ran.kernel_)

Learned kernel: 1.06**2 * Matern(length_scale=0.675, nu=2.5) + WhiteKernel(noise_level=0.248)


In [69]:
gp_random_split = {}
X_test_random = np.asarray(X_test_random, dtype=float)
gp_random_split['y_test_predic'],gp_random_split['std_test_predic'] = gpr_nt_ran.predict(X_test_random, return_std=True)


gp_random_split['mae'] = mean_absolute_error(gp_random_split['y_test_predic'], random_split['y_test'])
gp_random_split['rmse'] = np.sqrt(mean_squared_error(gp_random_split['y_test_predic'], random_split['y_test']))
gp_random_split['r2'] = r2_score(gp_random_split['y_test_predic'], random_split['y_test'])


print(f"MAE : {gp_random_split['mae']:.4g}")
print(f"RMSE: {gp_random_split['rmse']:.4g}")
print(f"R^2 : {gp_random_split['r2']:.4g}")

MAE : 7.241
RMSE: 7.832
R^2 : 0.7045


In [58]:
gp_random_split['x_space'] = np.asarray(comp_space[['Cu','Ni','Al','VEC','del_r','del_EN']], dtype=float)
gp_random_split['y_predic'], gp_random_split['std_y_predic'] = gpr_nt_ran.predict(gp_random_split['x_space'], return_std=True)

with open(PREDICTIONS_DIR/"GP-Experiments/gp_random_split", "wb") as f:
    pickle.dump(gp_random_split, f)

Experimental data GP - Kmeans splitting

In [72]:
with open(SAMPLING_DIR/"cluster_split", "rb") as f:
    cluster_split = pickle.load(f)

In [73]:
X_train_cluster = cluster_split['X_train_split'].drop(columns=['r','r_ave','S'])
X_test_cluster = cluster_split['X_test_split'].drop(columns=['r','r_ave','S'])

In [74]:
X_train = np.asarray(X_train_cluster, dtype=float)
y_train = np.asarray(cluster_split['y_train'], dtype=float).ravel()

kernel = (
    C(1.0, (1e-3, 1e3)) *
    Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e2), nu=2.5)
    + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-8, 1e1))
)

gpr_nt_clus = GaussianProcessRegressor(
    kernel=kernel,
    alpha=0.0,   
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

gpr_nt_clus.fit(X_train, y_train)
print("Learned kernel:", gpr_nt_clus.kernel_)

Learned kernel: 1.04**2 * Matern(length_scale=0.534, nu=2.5) + WhiteKernel(noise_level=0.204)


In [75]:
gp_cluster_split = {}
X_test_cluster = np.asarray(X_test_cluster, dtype=float)
gp_cluster_split['y_test_predic'],gp_cluster_split['std_test_predic'] = gpr_nt_clus.predict(X_test_cluster, return_std=True)


gp_cluster_split['mae'] = mean_absolute_error(gp_cluster_split['y_test_predic'], cluster_split['y_test'])
gp_cluster_split['rmse'] = np.sqrt(mean_squared_error(gp_cluster_split['y_test_predic'], cluster_split['y_test']))
gp_cluster_split['r2'] = r2_score(gp_cluster_split['y_test_predic'], cluster_split['y_test'])


print(f"MAE : {gp_cluster_split['mae']:.4g}")
print(f"RMSE: {gp_cluster_split['rmse']:.4g}")
print(f"R^2 : {gp_cluster_split['r2']:.4g}")


MAE : 8.338
RMSE: 11.39
R^2 : 0.3219


In [60]:
gp_cluster_split['x_space'] = np.asarray(comp_space[['Cu','Ni','Al','VEC','del_r','del_EN']], dtype=float)
gp_cluster_split['y_predic'], gp_cluster_split['std_y_predic'] = gpr_nt_clus.predict(gp_cluster_split['x_space'], return_std=True)

with open(PREDICTIONS_DIR/"GP-Experiments/gp_cluster_split", "wb") as f:
    pickle.dump(gp_cluster_split, f)

Adaptive kernel approach

In [ ]:
# Source and target variables for Adaptive transfer kernel

def data_transfer_gp (X_source,X_target, y_source, y_target):
    # Convert to numpy
    Xs = np.asarray(X_source, dtype=float)
    Xt = np.asarray(X_target, dtype=float)
    
    # Add domain indicator ( 0 = source, 1 = target)
    Xs_aug = np.c_[Xs, np.zeros((len(Xs),1))]
    Xt_aug = np.c_[Xt, np.ones((len(Xt),1))]
    
    #combine datasets
    X_train = np.vstack([Xs_aug, Xt_aug])
    y_train = np.concatenate([y_source, y_target])
    
    return X_train,y_train

In [80]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor

kernel = AdaptiveTransferKernel(
    kernel=1.0 * Matern(length_scale=1.0, nu=2.5),
    lamb=2.0,
    lamb_bounds=(1.0, 3.0),
    different_noises=False,
)

gpr_pe = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-5,            # jitter for numerical stability
    normalize_y=True,      # y standardization inside sklearn
    n_restarts_optimizer=3 # increase later (5-10) if slow/unstable
)